<a href="https://colab.research.google.com/github/nemo10-boop/AlphaToeXX/blob/main/MiniMaxToe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import math

print("✅ Ready")

In [ ]:
class Board:
    def __init__(self):
        self.grid = [" "] * 9      # 9 cells, index 0–8
        self.current_winner = None

    def display_board(self):
        for i in range(0, 9, 3):
            row = self.grid[i:i+3]
            print(f" {row[0]} | {row[1]} | {row[2]} ")
            if i < 6:
                print("---|---|---")

    def available_moves(self):
        return [i for i, spot in enumerate(self.grid) if spot == " "]

    def is_full(self):
        return " " not in self.grid

    def make_move(self, square, letter):
        if self.grid[square] == " ":
            self.grid[square] = letter
            if self.check_winner(square, letter):
                self.current_winner = letter
            return True
        return False

    def undo_move(self, square):
        self.grid[square] = " "
        self.current_winner = None

    def check_winner(self, square, letter):
        # Check row
        row_idx = square // 3
        if all(self.grid[row_idx*3 + i] == letter for i in range(3)):
            return True
        # Check column
        col_idx = square % 3
        if all(self.grid[col_idx + i*3] == letter for i in range(3)):
            return True
        # Check diagonals
        if square % 2 == 0:
            if all(self.grid[i] == letter for i in [0, 4, 8]):
                return True
            if all(self.grid[i] == letter for i in [2, 4, 6]):
                return True
        return False

    def is_game_over(self):
        return self.current_winner is not None or self.is_full()

# Test
b = Board()
b.display_board()
print("✅ Board ready")

In [ ]:
minimax_calls = 0  # track call count for performance comparison

def minimax(board, is_maximizing, ai_letter, human_letter):
    global minimax_calls
    minimax_calls += 1

    # Terminal states
    if board.current_winner == ai_letter:
        return 1
    if board.current_winner == human_letter:
        return -1
    if board.is_full():
        return 0

    if is_maximizing:
        best_score = -math.inf
        for move in board.available_moves():
            board.make_move(move, ai_letter)
            score = minimax(board, False, ai_letter, human_letter)
            board.undo_move(move)
            best_score = max(score, best_score)
        return best_score
    else:
        best_score = math.inf
        for move in board.available_moves():
            board.make_move(move, human_letter)
            score = minimax(board, True, ai_letter, human_letter)
            board.undo_move(move)
            best_score = min(score, best_score)
        return best_score

print("✅ Minimax ready")

In [ ]:
alphabeta_calls = 0  # track call count for comparison

def alphabeta(board, is_maximizing, ai_letter, human_letter,
              alpha=-math.inf, beta=math.inf):
    global alphabeta_calls
    alphabeta_calls += 1

    # Terminal states
    if board.current_winner == ai_letter:
        return 1
    if board.current_winner == human_letter:
        return -1
    if board.is_full():
        return 0

    if is_maximizing:
        best_score = -math.inf
        for move in board.available_moves():
            board.make_move(move, ai_letter)
            score = alphabeta(board, False, ai_letter, human_letter, alpha, beta)
            board.undo_move(move)
            best_score = max(score, best_score)
            alpha = max(alpha, best_score)
            if beta <= alpha:
                break  # ✂️ prune — no need to explore further
        return best_score
    else:
        best_score = math.inf
        for move in board.available_moves():
            board.make_move(move, human_letter)
            score = alphabeta(board, True, ai_letter, human_letter, alpha, beta)
            board.undo_move(move)
            best_score = min(score, best_score)
            beta = min(beta, best_score)
            if beta <= alpha:
                break  # ✂️ prune
        return best_score

print("✅ Alpha-Beta Pruning ready")

In [ ]:
def get_best_move(board, ai_letter, human_letter, use_pruning=True):
    best_score = -math.inf
    best_move  = None

    for move in board.available_moves():
        board.make_move(move, ai_letter)
        if use_pruning:
            score = alphabeta(board, False, ai_letter, human_letter)
        else:
            score = minimax(board, False, ai_letter, human_letter)
        board.undo_move(move)

        if score > best_score:
            best_score = score
            best_move  = move

    return best_move

# Quick test on empty board
test_board = Board()
move = get_best_move(test_board, "O", "X")
print(f"✅ AI chooses move: {move} (0-indexed) on empty board")

In [ ]:
def make_visual_board():
    buttons = []
    for i in range(9):
        btn = widgets.Button(
            description=" ",
            layout=widgets.Layout(width="80px", height="80px"),
            style={"button_color": "#f0f0f0", "font_size": "28px"}
        )
        buttons.append(btn)

    grid = widgets.GridBox(
        buttons,
        layout=widgets.Layout(
            grid_template_columns="repeat(3, 80px)",
            grid_gap="6px"
        )
    )

    status = widgets.Label(
        value="Your turn! You are ✕",
        style={"font_size": "16px"}
    )

    reset_btn = widgets.Button(
        description="🔄 Reset",
        layout=widgets.Layout(width="120px", height="36px"),
        style={"button_color": "#4CAF50"}
    )

    return buttons, grid, status, reset_btn

buttons, grid, status_label, reset_btn = make_visual_board()
print("✅ Visual board ready")

In [ ]:
HUMAN = "X"
AI    = "O"
game_board = Board()
game_active = True

def update_buttons():
    symbols = {" ": " ", "X": "✕", "O": "◯"}
    colors  = {" ": "#f0f0f0", "X": "#2196F3", "O": "#F44336"}
    for i, btn in enumerate(buttons):
        val = game_board.grid[i]
        btn.description = symbols[val]
        btn.style.button_color = colors[val]
        btn.disabled = (val != " ") or not game_active

def check_game_state():
    global game_active
    if game_board.current_winner == HUMAN:
        status_label.value = "🎉 You win! (Impressive!)"
        game_active = False
    elif game_board.current_winner == AI:
        status_label.value = "🤖 AI wins! (Expected — it's unbeatable)"
        game_active = False
    elif game_board.is_full():
        status_label.value = "🤝 It's a draw!"
        game_active = False

def ai_move():
    move = get_best_move(game_board, AI, HUMAN, use_pruning=True)
    if move is not None:
        game_board.make_move(move, AI)
    update_buttons()
    check_game_state()
    if game_active:
        status_label.value = "Your turn! You are ✕"

def on_button_click(btn):
    global game_active
    if not game_active:
        return
    idx = buttons.index(btn)
    if game_board.grid[idx] != " ":
        return

    # Human move
    game_board.make_move(idx, HUMAN)
    update_buttons()
    check_game_state()

    # AI responds
    if game_active:
        status_label.value = "🤖 AI is thinking..."
        ai_move()

def on_reset(btn):
    global game_board, game_active
    game_board  = Board()
    game_active = True
    status_label.value = "Your turn! You are ✕"
    update_buttons()

# Attach handlers
for btn in buttons:
    btn.on_click(on_button_click)
reset_btn.on_click(on_reset)

update_buttons()
print("✅ Game controller ready")

In [ ]:
display(
    widgets.VBox([
        widgets.Label("♟️ Tic-Tac-Toe — You (✕) vs AI (◯)",
                      style={"font_size": "20px"}),
        grid,
        status_label,
        reset_btn
    ])
)

In [ ]:
import matplotlib.pyplot as plt

results = {"States (Minimax)": [], "States (Alpha-Beta)": [], "Moves Made": []}

# Simulate from progressively filled boards
test_scenarios = [
    [],                    # empty board (9 moves left)
    [4],                   # center taken (8 moves left)
    [4, 0],               # 7 moves left
    [4, 0, 8],            # 6 moves left
    [4, 0, 8, 2],         # 5 moves left
]

for moves in test_scenarios:
    b1, b2 = Board(), Board()

    # Apply same moves to both boards
    letters = ["X", "O", "X", "O", "X"]
    for i, m in enumerate(moves):
        b1.make_move(m, letters[i])
        b2.make_move(m, letters[i])

    # Count Minimax calls
    minimax_calls = 0
    get_best_move(b1, "O", "X", use_pruning=False)
    mm_calls = minimax_calls

    # Count Alpha-Beta calls
    alphabeta_calls = 0
    get_best_move(b2, "O", "X", use_pruning=True)
    ab_calls = alphabeta_calls

    results["States (Minimax)"].append(mm_calls)
    results["States (Alpha-Beta)"].append(ab_calls)
    results["Moves Made"].append(len(moves))

# Plot
fig, ax = plt.subplots(figsize=(9, 5))
x = results["Moves Made"]
ax.plot(x, results["States (Minimax)"],  "o-", label="Minimax",        color="#F44336", linewidth=2)
ax.plot(x, results["States (Alpha-Beta)"], "s-", label="Alpha-Beta",   color="#4CAF50", linewidth=2)
ax.fill_between(x, results["States (Alpha-Beta)"], results["States (Minimax)"],
                alpha=0.1, color="green", label="Nodes pruned")

ax.set_xlabel("Moves already made on board")
ax.set_ylabel("Nodes evaluated")
ax.set_title("Minimax vs Alpha-Beta Pruning — Search Space Comparison")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

# Summary table
print(f"\n{'Moves Made':<12} {'Minimax':>12} {'Alpha-Beta':>12} {'Pruned %':>10}")
print("─" * 50)
for i, m in enumerate(results["Moves Made"]):
    mm = results["States (Minimax)"][i]
    ab = results["States (Alpha-Beta)"][i]
    pruned = round((1 - ab/mm) * 100, 1) if mm > 0 else 0
    print(f"{m:<12} {mm:>12} {ab:>12} {pruned:>9}%")

In [ ]:
import pickle

# Package everything the AI needs into one object
ai_package = {
    "get_best_move": get_best_move,
    "alphabeta":     alphabeta,
    "Board":         Board,
    "AI":            AI,
    "HUMAN":         HUMAN
}

with open("alphatoe_ai.pkl", "wb") as f:
    pickle.dump(ai_package, f)

print("✅ Saved as alphatoe_ai.pkl")

In [ ]:
from google.colab import files
files.download("alphatoe_ai.pkl")